# ANÁLISE BOX OFFICE | _The Numbers Domestic Box Office Dataset_
___
Análises voltadas para dados de arrecadação de bilheteria.
___

### 0 | Importações
___

In [2]:
import pandas as pd             # Manipulação de dados
import plotly.express as px     # Visualização de dados

In [3]:
df = pd.read_csv('../data/processed/tabela_tratada.csv') # Importando o dataset

### 1 | Arrecadação na estreia vs. Legs como influência no box office:
___

In [4]:
fig = px.scatter(
    df,
    y='tn_legs',          # Eixo X: Multiplicador de "pernas" (sustentabilidade do filme)
    x='domestic_bo',      # Eixo Y: Bilheteria doméstica ajustada pela inflação
    hover_name='nome',    # Exibe o nome do filme ao passar o mouse
    hover_data=['year', 'rating', 'estreia_bilheteria'], # Dados extras no tooltip
    color='genre',        # Opcional: Colorir os pontos por gênero (se a coluna existir)
    title='Relação entre Legs e Bilheteria Doméstica por gênero',
    labels={
        'tn_legs': 'TN Legs (Multiplicador)',
        'domestic_bo': 'Bilheteria Doméstica Ajustada ($)',
        'genre': 'Gênero'
    },
    opacity=0.75
)

# 3. Ajustes visuais (Opcional, mas altamente recomendado para bilheterias)
fig.update_layout(
    yaxis_title='TN Legs',
    xaxis_title='Bilheteria Doméstica Ajustada',
    #yaxis_type='log',     # Transforma o eixo Y em escala logarítmica para melhor visualização
    width=900,
    height=600,
    template='plotly_white'
)

# Exibe o gráfico interativo
fig.show()

In [5]:
# Remove valores nulos nas três colunas essenciais
df_plot = df.dropna(subset=['estreia_bilheteria', 'tn_legs', 'domestic_bo']).copy()

# 2. Cria o Gráfico de Bolhas
fig = px.scatter(
    df_plot,
    x='estreia_bilheteria',
    y='tn_legs',
    size='domestic_bo',       # O TAMANHO da bolha é o Box Office Total (quanto maior, mais faturou)
    color='domestic_bo',      # A COR da bolha reforça o volume de faturamento
    hover_name='nome',
    hover_data=['year', 'genre', 'rating'],
    title='Arrecadação na estreia vs. Legs como influência no Box Office Total',
    labels={
        'estreia_bilheteria': 'Bilheteria de Estreia ($)',
        'tn_legs': 'TN Legs (Sustentabilidade / Consistência)',
        'domestic_bo': 'Box Office Total ($)'
    },
    color_continuous_scale='Plasma', # Escala de cores (tons mais claros/amarelos = maiores bilheterias)
    size_max=60                      # Limite máximo do tamanho da bolha para não embolar
)

# 3. Ajustes visuais (Escala log no eixo X por conta da disparidade dos blockbusters)
fig.update_layout(
    #xaxis_type='log',
    xaxis_title='Bilheteria de Estreia (Escala Log)',
    yaxis_title='TN Legs (Multiplicador de Consistência)',
    width=1000,
    height=650,
    template='plotly_white'
)

# Exibe o gráfico interativo
fig.show()

O gráfico aparenta denunciar, em geral, a presena de um gradiente de cores da esquerda para a direita do gráfico. Isso indica que a bilheteria total está mais associada com a arrecadação na estreia do que com o valor de Legs de um filme.

In [6]:
df_plot = df.dropna(subset=['estreia_bilheteria', 'domestic_bo']).copy()

# Cria o gráfico de dispersão
fig = px.scatter(
    df_plot,
    x='estreia_bilheteria',
    y='domestic_bo',
    hover_name='nome',
    #color='genre',
    hover_data=['year', 'genre', 'tn_legs'],
    trendline='ols',          # Linha de tendência estatística
    title='Relação Direta: Bilheteria de Estreia vs. Bilheteria Doméstica Total',
    labels={
        'estreia_bilheteria': 'Bilheteria de Estreia ($)',
        'domestic_bo': 'Bilheteria Doméstica Total ($)'
    },
    opacity=0.6
)

# Ajustes vitais: Escala logarítmica em AMBOS os eixos para dados de bilheteria
fig.update_layout(
    #xaxis_type='log',
    #yaxis_type='log',
    xaxis_title='Bilheteria de Estreia',
    yaxis_title='Bilheteria Doméstica Total',
    width=900,
    height=600,
    template='plotly_white'
)

fig.show()

O gráfico acima parece confirmar a hipótese de uma relação linear entre arrecadação na estreia e bilheteria total.

### 2 | Influência da segunda semana na arrecadação da bilheteria
___

In [7]:
df_plot = df.dropna(subset=['estreia_bilheteria', 'semana_2', 'domestic_bo']).copy()

# 2. Calcula a Taxa de Retenção (Semana 2 / Estreia)
df_plot['retencao_semana_2'] = df_plot['semana_2'] / df_plot['estreia_bilheteria']

# Filtra eventuais distorções extremas de dados (ex: filmes com lançamento limitado que expandiram salas na semana 2)
df_plot = df_plot[(df_plot['retencao_semana_2'] > 0) & (df_plot['retencao_semana_2'] <= 2.0)]

# 3. Cria o Gráfico de Dispersão com Linha de Tendência
fig = px.scatter(
    df_plot,
    x='retencao_semana_2',
    y='domestic_bo',
    hover_name='nome',
    hover_data=['year', 'genre', 'estreia_bilheteria', 'semana_2'],
    trendline='ols',          # Linha de tendência estatística (requer statsmodels)
    title='Impacto da Retenção da 2ª Semana na Arrecadação Total',
    labels={
        'retencao_semana_2': 'Taxa de Retenção (Semana 2 / Estreia)',
        'domestic_bo': 'Box Office Doméstico Total ($)'
    },
    #color='genre',            # Colore os pontos por gênero para análise detalhada
    opacity=0.75
)

# 4. Ajustes visuais (Escala log no eixo Y para bilheterias)
fig.update_layout(
    xaxis_title='Taxa de Retenção da 2ª Semana (Menor queda = Direita)',
    yaxis_title='Box Office Total Ajustado ($ - Escala Log)',
    #yaxis_type='log',
    width=1000,
    height=650,
    template='plotly_white'
)

# Exibe o gráfico interativo
fig.show()

In [8]:
df_plot = df.dropna(subset=['semana_2', 'domestic_bo']).copy()

# Cria o gráfico de dispersão
fig = px.scatter(
    df_plot,
    x='semana_2',
    y='domestic_bo',
    hover_name='nome',
    #color='genre',
    hover_data=['year', 'genre', 'tn_legs'],
    trendline='ols',          # Linha de tendência estatística
    title='Relação Direta: Bilheteria da Segunda Semana vs. Bilheteria Doméstica Total',
    labels={
        'semana_2': 'Bilheteria da Segunda Semana ($)',
        'domestic_bo': 'Bilheteria Doméstica Total ($)'
    },
    opacity=0.6
)

fig.update_layout(
    #xaxis_type='log',
    #yaxis_type='log',
    xaxis_title='Bilheteria da Segunda Semana',
    yaxis_title='Bilheteria Doméstica Total',
    width=900,
    height=600,
    template='plotly_white'
)

fig.show()

### 3 | Evolução da média da arrecadação dos filmes ano a ano
___


In [9]:
df_anual = df.groupby('year')['domestic_bo'].mean().reset_index()

# Opcional: Se quiser remover eventuais anos nulos ou 0 que possam corromper a linha
df_anual = df_anual.dropna()

# 3. Cria o gráfico de linhas com Plotly Express
fig = px.line(
    df_anual,
    x='year',                  # Eixo X: Anos
    y='domestic_bo',           # Eixo Y: Arrecadação média
    markers=True,              # Adiciona pontos visíveis em cada ano (facilita o hover)
    title='Evolução da Arrecadação Média Doméstica por Ano (Ajustada pela Inflação)',
    labels={
        'year': 'Ano de Lançamento',
        'domestic_bo': 'Média de Bilheteria Doméstica ($)'
    }
)

# 4. Ajustes visuais para formatar o layout e os eixos
fig.update_layout(
    xaxis_title='Ano',
    yaxis_title='Arrecadação Média Ajustada ($)',
    width=1000,
    height=600,
    template='plotly_white',
    hovermode='x unified'      # Une o tooltip de hover para facilitar a leitura por ano
)

# Formata o eixo Y para exibir os valores de forma amigável (opcional)
fig.update_yaxes(tickprefix='$', separatethousands=True)

# Exibe o gráfico interativo
fig.show()